<p align="center" style="font-size:50px"><b>CS-566 Deep Reinforcement Learning</b></p>
<p align="center" style="font-size:22px"><b>Term Project: Bipedal Walker</b></p>
<p align="center" style="font-size:22px"><b>Course Instructor: Dr Nazar Khan</b></p>
<p align="center" style="font-size:22px"><b>Submitted by: Rida Shahid - MSCSF25M015</b></p>

**Step 1. Import/Install all necessary Dependencies**



In [1]:
pip install swig


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.3 MB/s eta 0:00:00


In [2]:
pip install "gymnasium[box2d]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.1 MB/s eta 0:00:00


In [3]:
!pip install 'stable_baselines3'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 19.8 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.3.0
    Uninstalling gymnasium-1.3.0:
      Successfully uninstalled gymnasium-1.3.0


**Step 2. Complete Pipeline from Training an agent to evaluating it.**

In [ ]:

# importing dependencies
import os
import glob
import torch
import numpy as np
import gymnasium as gym
from stable_baselines3 import SAC
from gymnasium.wrappers import RecordVideo


# Environment
env = gym.make("BipedalWalker-v3")

# building a model with optimal hyperparameters from the study.
#(i.e., O. Aydogmus, M. Yilmaz: Comparative Analysis of Reinforcement Learning Algorithms for Bipedal Robot Locomotion)
model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=2.52e-3,
        gamma=0.96667,
        tau=1.68e-1,
        verbose=1,
    )

# Training the model for 2 Million steps
model.learn(2_000_000)


# Setting up Evaluation Environment
eval_env = gym.make("BipedalWalker-v3", render_mode="rgb_array")

# Recording the video
eval_env = RecordVideo(
    eval_env,
    video_folder="bipedalwalker",
    name_prefix="eval",
    episode_trigger=lambda x: True
)


# Testing the trained agent
# No. of testing episodes
n_episodes = 10

# storing each episode return
episode_rewards = []

for episode in range(n_episodes):

    obs, _ = eval_env.reset()
    episode_over = False
    episode_reward = 0
    step_count = 0

    while not episode_over:

        action, _ = model.predict(obs,deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        episode_over = terminated or truncated
        episode_reward += reward. # adding reward to make return
        step_count += 1. # incrementing steps

    episode_rewards.append(episode_reward)
    mean_reward = np.mean(episode_rewards) #mean reward

    print(
        f"Episode Reward: {episode_reward:.2f} | "
        f"Mean Reward: {mean_reward:.2f} | "
        f"Steps: {step_count}"
    )

#closing testing environment
eval_env.close()



Streaming output truncated to the last 5000 lines.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 719      |
|    ep_rew_mean     | 313      |
| time/              |          |
|    episodes        | 1404     |
|    fps             | 45       |
|    time_elapsed    | 9727     |
|    total_timesteps | 1186340  |
| train/             |          |
|    actor_loss      | -10.8    |
|    critic_loss     | 0.179    |
|    ent_coef        | 0.00419  |
|    ent_coef_loss   | 1.88     |
|    learning_rate   | 0.00252  |
|    n_updates       | 1186238  |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 725      |
|    ep_rew_mean     | 316      |
| time/              |          |
|    episodes        | 1408     |
|    fps             | 45       |
|    time_elapsed    | 9791     |
|    total_timesteps | 1189246  |
| train/             |          |
|    actor_loss      | -11.2   

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/bipedalwalker folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode Reward: 325.56 | Mean Reward: 325.56 | Steps: 657
Episode Reward: 326.39 | Mean Reward: 325.98 | Steps: 659
Episode Reward: 325.62 | Mean Reward: 325.86 | Steps: 674
Episode Reward: 326.39 | Mean Reward: 325.99 | Steps: 647
Episode Reward: 326.11 | Mean Reward: 326.01 | Steps: 655
Episode Reward: 325.03 | Mean Reward: 325.85 | Steps: 663
Episode Reward: 324.71 | Mean Reward: 325.69 | Steps: 665
Episode Reward: 325.31 | Mean Reward: 325.64 | Steps: 665
Episode Reward: 324.95 | Mean Reward: 325.56 | Steps: 658
Episode Reward: 325.28 | Mean Reward: 325.54 | Steps: 662


**Step 3. Play the video on colab**

In [ ]:

# Play latest video inline

from IPython.display import HTML, display
from base64 import b64encode

latest = sorted(glob.glob("bipedalwalker/*.mp4"))[-1]

video = open(latest, "rb").read()

encoded = b64encode(video).decode()

display(
    HTML(
        f'''
        <video width="400" controls>
            <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        </video>
        '''
    )
)